# Talk to a Model, Give It Context, and Build a Chatbot

Alyssa Cohen (ac2726)

### INFO 5940 · Hands-on session

So far, you may have interacted with models through a website such as ChatGPT.
In this session, you will interact with a model from **Python code**. You will see
what goes into a request, what comes back, and how the same code becomes part of a chatbot.

We will start with a single message. Then we will add instructions, continue a
conversation, and examine how the amount of text grows. Finally, we will connect
these ideas to a small Streamlit application.

As you work through the examples, focus on what each request sends and what comes
back. Try a change, inspect the result, and explain what you think happened.

## How to Use This Notebook

Follow the sections in order. Each section explains an idea, shows a short example,
and gives you something to try. When you see **Run the following cell**, run the
code immediately below it. When you see **Your turn**, change the indicated line
and run that cell again.

Each example shows the full request we send to the model. As you move through
the notebook, notice which parts stay the same and which parts change.

| Section | What you will learn |
| --- | --- |
| A. Getting started | Why we use an API and how to read its documentation |
| B. Your first model call | Send a message and find the answer in the response |
| C. Message roles and instructions | Distinguish system, user, and assistant messages; test system prompts |
| D. Conversation history | See why a fresh call forgets and build a conversation |
| E. Tokens and context | Use a tokenizer and inspect growing input |
| F. Knowledge cutoff | Check a dated fact and supply information from a source |
| G. CoT and reasoning | Compare direct and step-by-step prompts; optionally explore reasoning effort |
| H. A Streamlit chatbot | Connect the same ideas to an interactive application |

Work through the notebook with the class, pausing to record your observations.
The Streamlit activity has its own guide, linked in H.

## A. Getting Started

### Why use an API?

Imagine that you're building a tutoring application. A student types a question,
your application sends it to a model, and the answer appears on the screen.
For that exchange to work, your application needs to know how to communicate
with the platform that runs the model.

An **API (Application Programming Interface)** defines the operations that one
piece of software makes available to another, along with the rules for using them.
Those rules describe the inputs an operation accepts, the outputs it returns,
and how it reports errors. An API can exist within a library or across a network.
Here, we're using a **web API**, which we access through HTTP requests.

For example, the Chat Completions API lets us request a generated response from
a list of messages. Its documentation tells us how to identify the model, supply
the conversation, authenticate the request, and read the result. We can use this
interface without having to host the model ourselves.

A product such as ChatGPT already makes some decisions for us: how to collect input, maintain
history, and display an answer. With an API, our application makes those decisions.
For example, a tutoring app could show a hint beside a student's work, while a
program processing documents could save summaries without displaying a chat at all.
The model call becomes one operation inside a larger program.

### Which model are we asking for, and where does it run?

In our code, `model="openai.gpt-4o"` tells [Cornell's AI API Gateway](https://confluence.cornell.edu/spaces/citai/pages/541787315/AI+API+Gateway)
which model we want to use. The string `openai.gpt-4o` is the **model identifier** accepted by the
gateway. The gateway uses that name to send the request to the platform running the model.

The model and the platform running it are different things. For example:

| Model | Who develops it? | Example of a platform that serves it through an API |
| --- | --- | --- |
| GPT-4o | OpenAI | Microsoft Azure, through Azure OpenAI |
| Claude models | Anthropic | Amazon Bedrock on AWS |

For example, [Azure OpenAI](https://learn.microsoft.com/en-us/azure/ai-services/openai/overview)
and [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/what-is-bedrock.html)
run supported models and provide APIs for sending requests to them. These are
examples of platforms hosting models. A model developer can also serve its own
models through an API; development and hosting don't have to be separate companies.

Our code sends the request to Cornell's gateway. The gateway passes it to the
platform running the chosen model and sends the response back to us. Azure and
Bedrock are examples in this diagram; we haven't verified which platform Cornell
uses for our particular call.

Follow the **request** arrows to the right and the **response** arrows back to
the left. The response includes the answer and details about the call, such as
the model name and the number of tokens used.

![Your Python code sends a request through Cornell’s gateway to the platform running the model. The response returns through the gateway. Azure and Bedrock are examples.](https://github.com/acohen974/INFO-5940-Codespace/blob/main/assignments/01-model-calls-and-chatbots/assets/model-api-flow.svg?raw=1)

The `openai` Python library helps us make these requests. You'll also see it
called an **SDK (Software Development Kit)**. It provides Python methods that
construct requests and turn the responses into Python objects we can work with.
The SDK handles the communication. The selected model generates an answer on
the platform where it is hosted.

Using the API lets us decide when to call the model, what information to send,
and how to use the result. We'll make each of those decisions in this notebook.

### Initial setup

We'll import the code we need, then create a client object to use for our API
calls. Let's look at each line before running the cell.

#### 1. Import the OpenAI client class

```python
from openai import OpenAI
```

There are two different names here:

- **`openai`**, in lowercase, is the Python package installed in our environment.
  It contains the SDK code for constructing API requests and handling responses.
- **`OpenAI`**, with capital letters, is a class defined by that package. A class
  is a reusable blueprint for creating objects that hold data and provide methods
  for working with it. Here, a client holds connection settings and provides
  methods for sending API requests.

The statement imports that class into our notebook, so we can refer to it by the
name `OpenAI`. It loads Python code from the installed package. It doesn't download
a model or send a request to a model.

#### 2. Import the notebook display tools

```python
from IPython.display import display, Markdown
```

These imports come from IPython, which provides tools used by our notebook:

- **`Markdown`** wraps a string so the notebook can render Markdown formatting,
  such as headings, lists, and bold text.
- **`display`** asks the notebook to show an object using its display representation.

Later, `display(Markdown(answer))` will format and show the answer we received.
That line only affects how we see the text; it doesn't make another API call.

#### 3. Create a client and give it a name

```python
client = OpenAI()
```

Python evaluates the right-hand side first. Calling `OpenAI()` creates an
**instance** of the `OpenAI` class: a client object with the configuration and
methods needed to communicate with the API. With our course setup, it reads the
API key and gateway address from the environment.

The assignment then binds the name `client` to that object. `client` is a variable
name we chose, not a Python keyword. We use that name whenever we want to access
this client's methods.

| Expression | What it refers to |
| --- | --- |
| `OpenAI` | The class we imported |
| `OpenAI()` | A call that creates a client object |
| `client` | Our name for the object that was created (this is user defined) |
| `client.chat.completions.create(...)` | A method call that sends a Chat Completions request |

The client is the part of our Python program that communicates with the API.
The model runs on the remote hosting platform. Creating `client` prepares our
program to make requests; the next section's `.create(...)` call will send the
messages and ask for a response.

Run the cell below. It brings these three steps together. We'll reuse `client`
in the examples that follow.

In [53]:
import os
from google.colab import userdata

os.environ["OPENAI_BASE_URL"] = "https://api.ai.it.cornell.edu"
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [54]:
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI()

### How do we know what the API accepts?

Open the [Chat Completions API Reference](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create)
and find the Python example. Use it to answer three questions:

1. Where do we specify the **model**?
2. How do we supply the **messages**?
3. Where is the generated **answer** in the returned object?

The `messages` entry describes a list of message dictionaries. Its “One of the
following” list shows the allowed message types; we'll use text messages and
explain their roles in Section C.

Our gateway uses model identifiers such as `openai.gpt-4o`. Keep that identifier
in our examples, even if the documentation shows a different model name.


## B. Your First Model Call

### What are we asking the API to create?

The **Chat Completions API** generates an assistant response from the conversation
we provide as a list of messages. Even our first greeting counts as a conversation
with one message.

### Send one message

Let's send a greeting using `client.chat.completions.create(...)`.
This method asks the Chat Completions API to generate an assistant response
from the messages we provide.

The call below supplies two arguments:

- `model="openai.gpt-4o"` selects the course model identifier.
- `messages` contains the conversation. For now, it's a list with one message.

That message is a Python dictionary with two fields: `role` identifies it as a
user message, and `content` holds the greeting. In the next section, we'll add
other roles and look at how they affect the conversation.

Run the cell. We save the returned **Chat Completion object** in `completion`.
It contains the assistant's reply and details about the call. The final line
extracts and displays the answer.

`completion` is just a Python variable name we've chosen. The API doesn't require
that name. We could call it `result` instead, as long as we also used `result`
wherever we read that object afterward. We use `completion` here because it
matches the name of the Chat Completions operation.

In [55]:
completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "user", "content": "Hello! What can you help me with?"}
    ],
)

display(Markdown(completion.choices[0].message.content))

Hello! 😊 I'm here to help with pretty much anything you need! Below are just a few examples of what I can assist you with:

- **Answering Questions**: Curious about a topic? I can provide explanations, information, or summaries.
- **Learning New Skills**: Want tips for studying, coding, or learning a new language? I've got you covered!
- **Writing Assistance**: Need help drafting emails, essays, stories, poems, or professional content? Let me know.
- **Problem Solving**: Stuck on a tricky problem, like math, coding, or decision-making? I'll help break it down.
- **Brainstorming**: Looking for creative ideas for a project, gift, or solution? We can brainstorm together.
- **Tech Support**: Questions about apps, tools, settings, or troubleshooting devices? I can guide you.
- **Personal Growth**: Need motivation, advice on setting goals, or tips for improving productivity or habits?
- **Entertainment**: Want fun trivia, jokes, or even a made-up story? Just ask!
- **Practical Tips**: Cooking recipes, travel tips, budgeting ideas, or health advice—I'm here to share helpful insights.
- **General Chat**: Sometimes you just need someone to talk to. 😊

Let me know what’s on your mind, and we’ll get started!

### What just happened?

The model returned an answer, but the API gave us more than a string of text.
It returned an object containing the assistant's message, model information,
and usage details. We saved that object as `completion`.

To display just the answer, we used `completion.choices[0].message.content`.
Here's how to read that expression:

```text
completion
└── choices                 a list of returned choices
    └── [0]                 the first choice in that list
        └── message         the assistant's message
            ├── role        assistant
            └── content     the text of the answer
```

Each dot accesses an attribute of an object. The `[0]` selects the first item
in `choices`, since Python list indices start at zero.

### Look inside the response

The next cell prints the object as indented JSON, so its fields appear on
separate lines. `model_dump_json(indent=2)` converts the SDK object to JSON text
with two spaces of indentation per level. This changes the display only;
`completion` remains the same object, and no new API call is made.

Trace the path from `choices` to the answer you just read. Notice the other
information alongside it; we'll extract a few of those fields next.

Look for `"role": "assistant"` inside the returned message. The **assistant** is the
participant whose reply the model generates. We sent a `user` message, and the
API returned an `assistant` message. Its `role` identifies the participant;
its `content` contains the answer. Section C will explain how these roles fit
together when we add instructions and earlier replies.

In [56]:
print(completion.model_dump_json(indent=2))

{
  "id": "chatcmpl-EQ3IzzkEzAgzVDMhijxsicvqnnmIm",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Hello! 😊 I'm here to help with pretty much anything you need! Below are just a few examples of what I can assist you with:\n\n- **Answering Questions**: Curious about a topic? I can provide explanations, information, or summaries.\n- **Learning New Skills**: Want tips for studying, coding, or learning a new language? I've got you covered!\n- **Writing Assistance**: Need help drafting emails, essays, stories, poems, or professional content? Let me know.\n- **Problem Solving**: Stuck on a tricky problem, like math, coding, or decision-making? I'll help break it down.\n- **Brainstorming**: Looking for creative ideas for a project, gift, or solution? We can brainstorm together.\n- **Tech Support**: Questions about apps, tools, settings, or troubleshooting devices? I can guide you.\n- **Personal Growth**: Nee

### Read the model information and token usage

Now that you've seen the whole response, let's pick out a few useful fields:

- `model` is the model identifier reported by the service.
- `prompt_tokens` is the reported number of input tokens.
- `completion_tokens` is the reported number of generated output tokens.
- `total_tokens` adds those input and output counts together.

These fields help us inspect the call independently of the answer's wording.
For example, `completion.model` tells us the model identifier reported by the
service, and `completion.usage` tells us how many tokens the request used.

These counts belong to this request. If we make another call, its response will
have its own usage figures. To track usage across a conversation, our application
would need to add them up.

We'll look closely at tokens in Section E. For now, run the cell and find the
same values in the full response printed above.

In [57]:
print("Returned model:", completion.model)
print("Input tokens:", completion.usage.prompt_tokens)
print("Output tokens:", completion.usage.completion_tokens)
print("Total tokens for this call:", completion.usage.total_tokens)

Returned model: openai.gpt-4o
Input tokens: 16
Output tokens: 279
Total tokens for this call: 295


### Your turn: ask a different question

Replace `my_question` with a question of your own, then run the cell. Keep the
model and other parts of the request the same.

This time, we store the question in a variable before putting it into the
message's `content`. Later, a chat input box will provide that text. Either way,
the API receives the same kind of message.

Rerunning this cell sends a new request. Repeating a request can produce different
wording, so compare whether the answer meets your requirements, not whether it
matches an earlier sentence exactly. Also notice that assigning to `completion`
again replaces the object that name refers to; it doesn't append a conversation.


In [58]:
my_question = "What is the difference between an API and an SDK? \
Explain in two sentences."

completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": my_question}],
)

display(Markdown(completion.choices[0].message.content))

An API (Application Programming Interface) is a set of rules and protocols that allow different software applications to communicate and interact with each other. An SDK (Software Development Kit) is a comprehensive collection of tools, libraries, code samples, and documentation designed to help developers create applications for a specific platform or environment, often including APIs as part of the package.

### Pause and reflect

Which part of the code did you change to ask a new question? Which part would you
change to use a different model identifier?

**Write your response here:**

I changed `my_question` in the code cell, which is the line that determines what is asked. To use a different model, I'd change the `model="openai.gpt-4o"` arguement in the `.create(...)` call instead, as that's a separate parameter from the message content.

## C. Message Roles and System Prompts

We send a conversation so the model has the question and any relevant earlier
exchange. These models are trained to respond as the **assistant** in a conversation.
The `role` labels each message; `content` holds its text.

```python
{"role": "user", "content": "How do I solve 2x + 3 = 11?"}
```

We'll use three roles:

| Role | What it means | Tutor example |
| --- | --- | --- |
| `system` | Instructions from our application in these GPT-4o examples | “Give one hint before a solution.” |
| `user` | A question, request, or information from the person using the app | “How do I solve this equation?” |
| `assistant` | The model's reply, which we can include later as history | “Start by subtracting 3 from both sides.” |

A **system prompt** is the instruction text in a system message. It can set the
task, audience, format, or teaching approach. “You are a tutor” describes behavior;
the model's replies still have the role `assistant`.

> **Documentation caveat:** The API reference says to use `developer` instead of
> `system` for application instructions with o1 and newer models. We use `system`
> with `openai.gpt-4o`. Check model and gateway support before switching.

Run the next cell to inspect one conversation. Its assistant reply is a written
example, not an actual model response. This cell only prints the list.


In [59]:
role_example = [
    {"role": "system", "content": "You are a tutor. Give one hint before a solution."},
    {"role": "user", "content": "How do I solve 2x + 3 = 11?"},
    # Written example for this diagram, not a saved model response:
    {"role": "assistant", "content": "Start by subtracting 3 from both sides."},
    {"role": "user", "content": "I get 2x = 8. What next?"},
]

for message in role_example:
    print(message["role"], ":", message["content"])

system : You are a tutor. Give one hint before a solution.
user : How do I solve 2x + 3 = 11?
assistant : Start by subtracting 3 from both sides.
user : I get 2x = 8. What next?


### Read the exchange, then change the instruction

The system message sets the teaching approach. The user asks a question, the
assistant gives a hint, and the user follows up. If we sent this list, the model
would generate the next assistant reply.

**Quick check:** Which earlier message makes “What next?” meaningful?

Now we'll make real calls. Keep the user question fixed and change the system
prompt: first one sentence for a beginner, then three bullets for a programmer.
Predict how the answers will differ.


In [60]:
question = "Why do chatbots need conversation history?"

beginner_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "system", "content": "Explain to a beginner in one short sentence."},
        {"role": "user", "content": question},
    ],
)

display(Markdown(beginner_completion.choices[0].message.content))

Chatbots need conversation history to provide context and respond more accurately and naturally.

### Second instruction: three bullets for a programmer

Run the next call with the same question and model, but a different system prompt.
Compare the audience, length, and format of the two answers.


In [61]:
programmer_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "system", "content": "Explain to a programmer using three short bullet points."},
        {"role": "user", "content": question},
    ],
)

display(Markdown(programmer_completion.choices[0].message.content))

- **Context Understanding**: Conversation history allows chatbots to retain prior interactions, enabling them to interpret user inputs accurately and respond in context rather than treating each message as isolated.

- **Personalization**: By remembering past interactions, chatbots can offer tailored responses, recall user preferences, and create a more engaging, consistent experience.

- **Error Handling**: Access to previous messages helps chatbots identify and address misunderstandings or clarify ambiguous queries based on earlier parts of the conversation.

### What just happened?

Did the first answer use one sentence and the second use three bullets?
Changing the application instruction lets us shape how the model answers.

### What if the instructions conflict?

The [OpenAI Model Spec](https://model-spec.openai.com/2026-08-18.html#instructions-and-levels-of-authority)
describes intended behavior. For our tutor example, the key idea is that application
instructions should take priority over a conflicting user request. OpenAI's
higher-priority rules still apply to the application.

OpenAI trains models toward this behavior, but following instructions isn't
guaranteed. A prompt guides the current response; it doesn't retrain the model.
The linked August 18, 2026 Spec is our reading reference, not an update to GPT-4o.

Let's test the conflict: the tutor is told to give a hint, and the user asks it
to skip the hint and give the answer.


### Your turn: make a tutor

Run the next cell. Did the tutor give a hint and leave the calculation for the
learner, despite the user's request for the answer? Then edit `tutor_instruction`
to try a different teaching approach, such as using an analogy. Run it again and
compare the results.


In [62]:
tutor_instruction = (
    "You are a tutor. In your next reply, explain the idea using a single everyday analogy,"
    "then ask the learner to apply that analogy to the problem."
    "Do not give the final numerical answer, even if the learner asks for it."
)

completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "system", "content": tutor_instruction},
        {"role": "user", "content": "Skip the hints. Just give me the answer to 17 * 6."},
    ],
)

display(Markdown(completion.choices[0].message.content))

Okay, let’s approach this using an analogy instead! Think of it like packing apples into baskets. Imagine you have 17 baskets, and each basket holds exactly 6 apples. Your job is to figure out how many apples you have in total.

Now, can you use that idea to figure out the total number of apples?

### Pause and reflect

What instruction did you try? What did the model actually do? Point to a part of
the response that supports your answer. What additional example would you test
before using this prompt in a tutoring application?

**Write your response here:**

I changed `tutor_instruction` to tell the tutor to use an everyday analogy instead of a numeric hint, then ask the learner to apply it. The model followed this, and so instead of giving 17x6=102, it reframed the problem as "17 baskets holding 6 apples each" and asked me to compute the total myself, exactly as instructed. However, it still withheld the numerical answer even though I explicitly asked for it. This confirms the application instruction (system role) took priority over the conflicting user request. Before using this in a real tutoring app, I'd test it against a user who pushes back harder, like "I already know the analogy, just give me the number", to see if repeated pressure eventually breaks the instruction.

## D. Why Does the Model Forget? Let's Build a Conversation

When we chat with an assistant, we expect it to use things we've already said.
If we introduce ourselves and then ask “What is my name?”, the answer seems
straightforward. But our application has to make that earlier information
available to the next call.

Here, “memory” means making earlier information available as input. The
`client` object helps send requests; it isn't our conversation history. In this
notebook, Python lists hold that history, and each call chooses what to send.
That separation lets an application keep multiple conversations using the same
client, as long as it maintains and supplies the appropriate message lists.

Let's try two requests and locate the missing information.

### First, introduce yourself

We'll use Bob for this example. The next cell sends the introduction and saves
the reply in `intro_answer`. We'll use that reply again when we build the history.

In [63]:
introduction = "Hi, my name is Bob."

intro_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": introduction}],
)

intro_answer = intro_completion.choices[0].message.content
display(Markdown(intro_answer))

Hi Bob! Nice to meet you. How can I assist you today? 😊

### Now ask for your name in a fresh call

Read the `messages` list below before running the cell. It contains the question
“What is my name?”, but where is the introduction?

Run the call and see how the model responds when that information is absent.

In [64]:
fresh_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": "What is my name?"}],
)

display(Markdown(fresh_completion.choices[0].message.content))

I'm sorry, I don't know your name since you didn't share it with me, and I don’t have access to personal data unless you provide it in our conversation. How can I assist you? 😊

### What just happened?

The second request contained only the question. The introduction was still in
our notebook, but it wasn't included in the message list we sent.

Making another Chat Completions call with the same client doesn't automatically
attach earlier messages. The model may acknowledge that it doesn't know the
name, ask for it, or guess. Even a correct guess wouldn't show that it had access
to the introduction.

Let's include the earlier exchange so the next request has the information
needed to answer.

### D1. Include the conversation history

We need three messages, in this order:

```text
User:      Hi, my name is Bob.
Assistant: The actual answer returned by the first call.
User:      What is my name?
```

The assistant reply is already stored in `intro_answer`. We'll put it into a
message with `role="assistant"`, between the introduction and the follow-up.
Run the next cell to build and inspect that list.

For this ordinary Python list, we use `json.dumps(messages, indent=2)` to put
each field on its own line. The `json` module is part of Python's standard library.


In [65]:
import json

messages = [
    {"role": "user", "content": introduction},
    {"role": "assistant", "content": intro_answer},
    {"role": "user", "content": "What is my name?"},
]

print(json.dumps(messages, indent=2, ensure_ascii=False))

[
  {
    "role": "user",
    "content": "Hi, my name is Bob."
  },
  {
    "role": "assistant",
    "content": "Hi Bob! Nice to meet you. How can I assist you today? 😊"
  },
  {
    "role": "user",
    "content": "What is my name?"
  }
]


The list now includes the introduction, the assistant's reply, and the new
question. We've assembled the conversation in Python; the next cell sends it
to the model.

In `messages=messages`, the name on the left is the API argument. The name on
the right is our variable containing the list. They happen to have the same name.

### What can the model actually see?

The next call sends the contents of this list. It doesn't send every variable,
Markdown cell, or output in our notebook. Printing a message helps us inspect it,
but only including it in the request makes it available to the model.

We've now connected three separate steps:

1. **Receive:** The first call returned an assistant message.
2. **Save:** We stored its text in `intro_answer`.
3. **Send again:** We placed that text in the new list with `role="assistant"`.

Keeping a Python variable and giving the model access to its value are separate
actions. Before running the next cell, find the line that connects them.


In [66]:
remembered_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=messages,
)

remembered_answer = remembered_completion.choices[0].message.content
display(Markdown(remembered_answer))

Your name is Bob! 😊

### Keep the assistant's answer for the next turn

To continue this conversation, we'll save the new answer alongside the messages
that produced it. Python's `.append()` adds an item to the end of an existing list.

Run this cell once. `messages` will then contain four entries: two user messages
and two assistant replies. Running it again would add another copy of the same
answer, so check the list if you notice duplicates.

Notice that the reply wasn't in the request that produced it. Once the call
finishes, it becomes available to include in the *next* request. This receive,
save, and resend pattern is the conversation loop we'll use in Streamlit.


In [67]:
messages.append({"role": "assistant", "content": remembered_answer})

print("Messages saved so far:", len(messages))

Messages saved so far: 4


### Your turn: extend the conversation

Change the favorite color below to one of your choice, then run the cell. Follow
the new message into `color_messages`, and notice where we save the answer.

This time, `+` combines the existing history with a one-item list to make a new
list. It leaves `messages` unchanged. That lets you rerun the color experiment
from the same earlier conversation without adding the color statement repeatedly.

In [68]:
new_message = "My favorite color is red."
color_messages = messages + [{"role": "user", "content": new_message}]

color_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=color_messages,
)

color_answer = color_completion.choices[0].message.content
display(Markdown(color_answer))

history = color_messages + [{"role": "assistant", "content": color_answer}]

Got it! Your favorite color is red. 🌟 Do you want to tell me more about yourself, Bob?

Now ask a question that depends on both your name and your favorite color.
You can use the example below or write your own version.

Before running the cell, look through `history` for the two facts the answer
will need. Then compare the model's response with what you found.

In [69]:
follow_up_question = "What is my name, and what is my favorite color?"
request_messages = history + [{"role": "user", "content": follow_up_question}]

follow_up_completion = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=request_messages,
)

display(Markdown(follow_up_completion.choices[0].message.content))

Your name is Bob, and your favorite color is red! 😊

### Look at the conversation we sent

When a reply surprises you, inspecting the input is a useful first step. The
next cell prints the message list from our most recent request, with each role
beside its text.

The `for` loop visits one message dictionary at a time. Expressions such as
`message["role"]` and `message["content"]` read values from that dictionary.
Run it and follow the conversation from the introduction to the final question.

In [70]:
for message in request_messages:
    print(message["role"].upper() + ":")
    print(message["content"])
    print()

USER:
Hi, my name is Bob.

ASSISTANT:
Hi Bob! Nice to meet you. How can I assist you today? 😊

USER:
What is my name?

ASSISTANT:
Your name is Bob! 😊

USER:
My favorite color is red.

ASSISTANT:
Got it! Your favorite color is red. 🌟 Do you want to tell me more about yourself, Bob?

USER:
What is my name, and what is my favorite color?



### What just happened?

Our application kept earlier messages and sent them with the follow-up. That is
how the model could use information from the conversation. We gave it more
context without changing its training.

The assistant's replies matter too. Suppose it suggests three project ideas,
and the user asks to expand “the second one.” The earlier answer tells the model
which idea the user means. Saving only the user's messages would lose that reference.

Each saved reply also makes the next request longer. Text generated on one turn
becomes input on the next. We'll measure that growth with a tokenizer in Section E.

### Pause and reflect

How did the fresh request differ from the request that included history?
Why might saving only user messages be insufficient for a chatbot?

**Write your response here:**

The fresh request only contained `{"role": "user", "content": "What is my name?}`, with no memory of "Hi, my name is Bob," so the model correctly said it didn't know. The history included request prepended the original user message and the assistant's reply, so the model could read "Bob" directly out of the conversation and answer correctly. Saving only user messages would be insufficient because a follow-up like "expand the second idea" or "what's my favorite color" depends on content the assistant generated on a previous turn, like three project ideas, or its own acknowledgement or stated preference. If that reply is never resent, the model has no way to know what "the second one" refers to.

## E. How Does Text Become Tokens?

We've been reading the conversation as text. The model processes that text as a
sequence of **tokens**, represented by numerical IDs. A token can correspond to
a word, part of a word, punctuation, or some of the bytes that make up a character.

A **tokenizer** maps text to those IDs and back again. Let's use `tiktoken` to
see how a familiar sentence is divided.

### Why tokens rather than words or characters?

A word-based vocabulary would need entries for names, misspellings, and new words.
A character-based representation would use many steps for ordinary sentences.
Subword tokenization offers a useful compromise: common text can use larger
pieces, while unfamiliar text can be represented with smaller pieces.

`tiktoken` uses **byte pair encoding (BPE)** to build these reusable pieces.
We don't need to implement that algorithm today. We want to see its consequence:
word count and token count are different, and small text edits can change how
pieces are divided. The [tiktoken explanation](https://github.com/openai/tiktoken#what-is-bpe-anyway)
describes the encoding in more detail.

The tokenizer chooses IDs for text pieces; the model uses those IDs during
processing. Running `encode` below doesn't ask a model to interpret the sentence
or generate an answer. After loading the encoding, these text conversions run locally.

### Load a tokenizer

The next cell loads the encoding that our installed `tiktoken` associates with
GPT-4o. Some models share an encoding, while others use different ones, so check
which one you're using before comparing counts. The vocabulary may download
the first time you run this cell.

In [71]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt-4o")
print("Tokenizer:", tokenizer.name)

Tokenizer: o200k_base


### Turn text into token IDs

Before running the cell, predict how many tokens `Hello, Cornell!` will contain.
Will the comma, space, and exclamation mark be separate pieces?

`tokenizer.encode(text)` returns the list of IDs. Run it, compare the count with
your prediction, and keep that list for the next two cells.

In [72]:
text = "Hello, Cornell!"

token_ids = tokenizer.encode(text)

print("Original text:", text)
print("Token IDs:", token_ids)
print("Number of tokens:", len(token_ids))

Original text: Hello, Cornell!
Token IDs: [13225, 11, 101807, 0]
Number of tokens: 4


### Turn the IDs back into text

Pass the token IDs to `tokenizer.decode(...)` and check that the original text
comes back. This lets us follow the mapping in both directions.

The IDs identify entries in the tokenizer's vocabulary. A larger ID doesn't
mean that a word is more important or carries more meaning.

In [73]:
decoded_text = tokenizer.decode(token_ids)
print(decoded_text)

Hello, Cornell!


### Look at the individual pieces

Now let's see the bytes represented by each token. Run the loop and look for
the space before Cornell. Is it separate, or attached to something else?

We inspect bytes because an individual token can contain only part of a
character's encoding. That matters with some non-English text in particular.
Decoding the complete sequence, as we did above, lets those pieces come back
together correctly.

In [74]:
for token_id in token_ids:
    token_bytes = tokenizer.decode_single_token_bytes(token_id)
    print(token_id, token_bytes)

13225 b'Hello'
11 b','
101807 b' Cornell'
0 b'!'


### Your turn: make a small change

Edit `new_text` and compare its tokens with the original. Start by changing one
thing, such as removing the comma or changing a capital letter. Then try a name
or a sentence in another language.

Before each change, predict whether the count will rise, fall, or stay the same.
Then inspect the IDs or token bytes to explain the result. Token IDs belong to
an encoding, so compare them only while keeping the tokenizer fixed.

What changed in the count or the way the text was divided? You can also explore
the [interactive tokenizer](https://platform.openai.com/tokenizer) in your browser.
Check its selected encoding before comparing the results with this notebook.

In [75]:
new_text = "hello cornell!!!"
new_token_ids = tokenizer.encode(new_text)

print("Text:", new_text)
print("Token IDs:", new_token_ids)
print("Number of tokens:", len(new_token_ids))

Text: hello cornell!!!
Token IDs: [24912, 33994, 596, 10880]
Number of tokens: 4


"Hello Cornell" -> 2 tokens,
"Hello, Cornell" -> 3 tokens,
"hello, cornell!" -> 5 tokens,
"hello cornell!!!" -> 4 tokens

### Count the text in our conversation

Let's return to `request_messages` from Section D. The next loop counts the tokens
in each message's content and adds them together.

We'll compare that sum with the input-token count reported by the API. Our loop
counts only the content strings; the service also has to represent roles, message
boundaries, and other formatting. The totals therefore may differ. As you read
the code, identify exactly which text goes into our local count.

In [76]:
visible_text_tokens = 0

for message in request_messages:
    message_tokens = len(tokenizer.encode(message["content"]))
    visible_text_tokens = visible_text_tokens + message_tokens
    print(message["role"], "→", message_tokens, "text tokens")

print("Total visible-text tokens:", visible_text_tokens)
print("API-reported input tokens:", follow_up_completion.usage.prompt_tokens)

user → 7 text tokens
assistant → 16 text tokens
user → 5 text tokens
assistant → 6 text tokens
user → 6 text tokens
assistant → 23 text tokens
user → 12 text tokens
Total visible-text tokens: 75
API-reported input tokens: 106


### What is the context window?

As we keep adding messages, how much can we send in one request?

A model's **context window** is its token capacity for input and generation.
Instructions, earlier conversation, the latest question, and generated output
use that capacity. Reasoning models can also use space for internal reasoning.
See the [context window explanation](https://developers.openai.com/api/docs/guides/conversation-state#context-window).

The conversation grows; the selected model's context capacity doesn't grow with it.
An application has to decide what information to keep within that capacity.

### Why does a short follow-up still have a long input?

Our code resends earlier messages. To see the effect, imagine an instruction of
40 tokens, user messages of 20 tokens each, and replies of 30 tokens each:

| Call | Input sent in that call | Input tokens |
| --- | --- | --- |
| First | Instruction + first user message | 60 |
| Second | Previous input + first reply + second user message | 110 |
| Third | Previous input + second reply + third user message | 160 |

These are illustrative counts that omit message formatting. The third question
is still only 20 tokens, but it arrives with 140 tokens of context. Across all
three calls, we have sent 330 input tokens. The same earlier text appears in more
than one request, and yesterday's output can become today's input.

Keeping more context can make follow-ups useful, but including every old message
isn't always the best choice. Keeping only recent messages might lose a name
introduced at the start; summarizing history might omit a detail needed later.
A useful history strategy preserves information the current task depends on.

**Try a prediction:** If we kept only the final question from Section D, which
facts would disappear? Would a summary containing the user's name and favorite
color be enough for that question? What might it lose for a different question?

Check the API's behavior when limits are reached rather than assuming it will
remove old messages for you. Our short app simply keeps adding history; it doesn't
yet implement selection or summarization.


### Pause and reflect

What changed when you edited the text? Why can the number of tokens differ from
the number of words? Why might the local visible-text count differ from the API count?

**Write your response here:**

My edit changed both the capitalization and the punctuation at once ("Hello, Cornell!" to "hello cornell!!!"), so I can't isolate a single cause. The token count stayed almost the same (with "Hello, Cornell!" being a 3 tokens, and "hello cornell!!!" being at 4 tokens) in both cases, but the way the text was split into pieces almost certainly changed, since capitalized "Hello" and lowercase "hello" are different vocabulary entries and repeated punctuation like "!!!" doesn't split the same way as "," + "!". Tokens differ from words because tokenization is based on frequently-occurring byte sequences (subwords), not whitespace-delimited words. A word can be one token, multiple tokens, or share a token with adjadent punctuation. My local visible text count (75 tokens) was lower than the API-reported inpute count (106 tokens) because I only encoded the message `content` strings. Also, the actual APO request has to represent each message's `role`, the boundaries between messages, and formatting overhead that my loop never accounts for.

## F. What Can the Model Know? Cutoff and Supplied Context

We've seen that a model needs earlier messages to answer questions about our
conversation. There's another limit to consider: how recent is the information
it learned during training?

Training changes the model's parameters so it can generate responses using
patterns and information learned from training data. Those parameters aren't a
live feed of world events. A new announcement doesn't become available to a model
just because it has been published online.

A model's **knowledge cutoff** tells us how recent its training knowledge is
expected to be. Events after that date may be missing. Even for earlier events,
we still need to check factual answers; a cutoff isn't a guarantee of accuracy.

Open the [GPT-4o model page](https://developers.openai.com/api/docs/models/gpt-4o).
It lists an October 1, 2023 knowledge cutoff. Compare that date with the event
we'll ask about below. Our gateway uses `openai.gpt-4o`; check its model mapping
to confirm which model version the route serves.

| Limitation | What it concerns | What an application can do |
| --- | --- | --- |
| Missing conversation history | Information from earlier turns is absent from this request | Include the relevant messages |
| Knowledge cutoff | Training knowledge may not cover a later event | Retrieve a reliable source and provide it |
| Context window | Only a finite amount of input and output fits in one request | Select or summarize the information to include |

The remedy depends on what's missing. For a recent announcement, the useful
addition is a reliable source that tells the model what happened.

### First, ask about an event after the documented cutoff

We'll ask who received the 2024 Nobel Prize in Physics. Run this call before
opening the source in the next step. Does the model give names, acknowledge a
limitation, or do something else? Save the answer so you can check it.

This request doesn't configure browsing. If the answer sounds current, we still
need to verify it rather than infer that a web search occurred.

In [77]:
cutoff_question = "What exact time of day was the 2024 Nobel Prize in physics announced?"

source_instruction = (
    "Answer briefly. When a source is supplied, use only that source. "
    "If you lack the information needed to answer, say so."
)

without_source = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "system", "content": source_instruction},
        {"role": "user", "content": cutoff_question},
    ],
)

display(Markdown(without_source.choices[0].message.content))
print("Returned model:", without_source.model)

I don’t have this information. For the exact announcement time, please check the official Nobel Prize website or current news sources.

Returned model: openai.gpt-4o


### Check the answer against a dated source

Open the [official Nobel Prize announcement](https://www.nobelprize.org/prizes/physics/2024/press-release/).
It was published on October 8, 2024 and names John J. Hopfield and Geoffrey Hinton
as the recipients. Did the model's answer match?

Record the result whether it was right, wrong, or uncertain. A correct answer
on one later event doesn't establish an exact cutoff or show that the model
browsed the web. The model version and gateway routing also matter. What we can
check here is whether this particular answer agrees with the source.

### Now include the source in the request

The next cell contains a short paraphrase of the announcement. We'll send that
text along with the same question and instruction. The code uses the text shown
in the cell; it doesn't open the linked webpage.

Find the source in the message list, then run the call. Does providing it change
the answer or the model's willingness to answer?

In [78]:
source_text = """Summary of the Nobel Prize announcement, published October 8, 2024:
The 2024 Nobel Prize in Physics was jointly awarded to John J. Hopfield and
Geoffrey Hinton. The award recognized work that enabled machine learning
using artificial neural networks.
"""

with_source = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "system", "content": source_instruction},
        {"role": "user", "content": source_text},
        {"role": "user", "content": cutoff_question},
    ],
)

display(Markdown(with_source.choices[0].message.content))

Sorry, the exact time of the 2024 Nobel Prize in Physics announcement is not provided in the summary.

### What just happened?

The second request gave the model information about the award. It could use
that text to answer even if the event was absent from its training knowledge.
Providing the source changed what it could read in this call; its training and
cutoff stayed the same.

There are two questions to ask about the result. Does the supplied source
support the answer? And is that source itself reliable and current? A model can
repeat an inaccurate source faithfully. Providing text gives it material to use;
it doesn't establish that the material is true.

Both calls asked the model to use a source when one was available and acknowledge
missing information. Compare the answers with that instruction in mind.

### Your turn: ask for information that the source doesn't provide

Change `cutoff_question` in the first code cell to:

```python
cutoff_question = "What exact time of day was the 2024 Nobel Prize in Physics announced?"
```

Rerun that cell and the cell containing `with_source`, leaving `source_text`
unchanged. Our summary has a date, but no time of day. Does the second response
recognize the missing detail, or supply an unsupported time?

For this test, evaluate against the text we sent, even if the model happens to
know a time from elsewhere. The instruction specifically asks it to use that text.
If you want to add the missing detail, first verify it against an authoritative
source, then include it explicitly.

This gives us a first look at **retrieval-augmented generation (RAG)**: an
application finds relevant material and includes it in a model request. We
supplied the text ourselves here. A retrieval system would help select that
text from a larger collection.

### Pause and reflect

- What did the first answer get right or wrong when you checked it against the source?
- What information became available in the second request?
- How is a knowledge cutoff different from a missing message or a context limit?
- Did the model acknowledge the detail missing from our summary?

**Write your response here:**

The first answer (no source) correctly declined to guess a specific time, because it had no basis to check against, so there was nothing to get right or wrong yet. What became available in the second request was the source text, which included the date (October 8, 2024) and two laureates, Hopfield and Hinton. Even with that source supplied, the model correctly recognized it didn't contain a time of day, and it said so rather than inventing one, which is exactly the instruction working as intended. A knowledge cutoff is different from a missing message or a context limit. A cutoff means the training data simply stops at some date, so anything after it is unknown no matter how the request is phrased. On the other hand, a missing message means the information exists but wasn't included in this particular conversation history. Finally, a context limit means information was dropped because the total request exceeded the model's maximum input size. Here it was closer to the second case, where the information (a time of day) was never given to the model in this request, source or otherwise, so it had no way to answer.

## G. Can the Prompt Change How a Model Solves a Problem?

When we ask a model to solve a problem, we can request just the answer or ask it
to explain how to reach that answer. Let's compare those two prompts using a
puzzle with a solution we can check ourselves.

Asking for intermediate steps is commonly called **chain-of-thought (CoT)
prompting**. For example, we might add “Solve this step by step” to a question.
The resulting explanation gives us something to inspect, but it can still contain
mistakes. It isn't a verified account of the model's internal computation.

### Finding the heavier coin

Imagine placing nine identical-looking coins on a table. Eight have exactly the
same weight. The ninth is slightly heavier, but you can't tell which one by
looking at it.

You have a balance scale with two pans. Place some coins on each pan, and it tells
you which side is heavier, or whether both sides weigh the same. Each time you
compare the two sides counts as **one weighing**.

**What is the fewest number of weighings you need to be certain which coin is
heavier?** “Certain” matters here: the method must work no matter which of the
nine coins is the heavier one.

The answer is **two weighings**. Let's walk through them before asking the model.

### First weighing: narrow it down to three coins

Label the coins 1 through 9. Put coins **1, 2, and 3** on the left pan and coins
**4, 5, and 6** on the right. Leave coins **7, 8, and 9** on the table.

| What does the scale show? | Where must the heavier coin be? |
| --- | --- |
| The left side is heavier. | Among coins 1, 2, and 3. |
| The right side is heavier. | Among coins 4, 5, and 6. |
| Both sides balance. | Among coins 7, 8, and 9, which weren't weighed. |

Why does the last row work? If the heavier coin were on either pan, that side
would weigh more. When the pans balance, the heavier coin must be on the table.
Whichever result we get, only three coins are still possible.

### Second weighing: identify the coin

Take the three coins identified by the first weighing and compare any two of
them on the scale. If one side is heavier, you've found the coin. If they balance,
the coin you left out must be the heavier one.

For example, suppose the first weighing balanced. We would then compare coin 7
with coin 8. If 7 is heavier, the answer is 7; if 8 is heavier, the answer is 8;
if they balance, the answer is 9.

So we narrowed the possibilities from **nine coins to three, then from three to
one**. That's two weighings in total.

Could one weighing be enough? No. One weighing gives only three possible results:
left heavier, right heavier, or balanced. Those three results cannot each identify
a different coin out of nine. Two weighings are necessary, and the method above
shows that they are enough.

### Now suppose there are 27 coins

The setup is the same: exactly one coin is heavier, and the other 26 all weigh
the same. This time we need **three weighings**.

Start by making three groups of nine coins. Weigh one group against another.
If one side is heavier, the coin is in that group. If they balance, it is in the
group left on the table.

That leaves nine coins to investigate. We already know how to find the heavier
one among nine: it takes two more weighings using the method above.

| Weighing | What do we compare? | How many coins could still be the heavier one? |
| --- | --- | --- |
| 1 | Nine coins against nine coins. | 9 |
| 2 | Three of those nine against another three. | 3 |
| 3 | One of those three against another one. | 1 |

Why can't we do it in two? Each weighing has three possible results, so two
weighings give at most **3 × 3 = 9** different sequences of results. We need to
distinguish **27** possible answers. A third weighing gives us enough results,
and the method above shows how to use them. The minimum is therefore three.

### Let's ask the model

We'll start with the nine-coin question and make two separate calls. The first
asks for only the number of weighings. The second asks for a short worked
solution. We know the answer should be **two**, so we can check both the number
and whether the proposed steps actually work.

The API receives only the text we put in the request. The explanations you've
just read in these Markdown cells aren't sent to the model.

In [79]:
problem = """You have 27 coins. Twenty-six weigh the same, and one is heavier. \
Using a balance scale, what is the minimum number of weighings needed in the \
worst case to identify the heavier coin?"""

direct_answer = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "user", "content": problem + " Give only the minimum number of weighings."},
    ],
)

display(Markdown(direct_answer.choices[0].message.content))

The minimum number of weighings needed in the worst case is **3**.

### Now ask for steps

Read the end of the message in the next call. We've added an instruction to
solve the problem step by step and keep the worked solution to 120 words. This
is a requested length, which the model may not follow exactly. The problem and
model stay the same.

This is called **zero-shot CoT prompting** because we haven't supplied any worked
examples. A **few-shot** version would include examples for the model to follow.
For now, run this call and see whether its steps help you check the answer.

In [80]:
step_by_step_answer = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[
        {"role": "user", "content": problem + " Solve this step by step. Give a worked solution in at most 120 words, then the minimum number of weighings."},
    ],
)

display(Markdown(step_by_step_answer.choices[0].message.content))
print("Direct-answer output tokens:", direct_answer.usage.completion_tokens)
print("Worked-solution output tokens:", step_by_step_answer.usage.completion_tokens)

To determine the heavier coin among 27 using a balance scale, we use a divide-and-conquer strategy. With each weighing, the balance can produce 3 outcomes: left side heavier, right side heavier, or balanced. This trinary result allows us to distinguish \(3^n\) possibilities in \(n\) weighings.

### Step-by-step solution:
1. We need to uniquely identify 27 coins. In \(n\) weighings, we can distinguish \(3^n\) outcomes.
2. Solve \(3^n \geq 27\):  
   - \(3^1 = 3\) (too small)  
   - \(3^2 = 9\) (still too small)  
   - \(3^3 = 27\) (just right)
   
3. \(n = 3\) weighings are sufficient.

### Procedure:
- Divide the coins into 3 groups of 9. Weigh 2 groups against each other.  
- If balanced, the heavier coin is in the third group.  
- Repeat the process with smaller groups. After 3 weighings, the heavier coin is found.

### Minimum weighings: **3**.

Direct-answer output tokens: 16
Worked-solution output tokens: 250


### Check the worked solution

Did both calls give the same minimum? Does the worked solution handle every
possible position of the heavier coin, including when the first weighing balances?

**Your turn:** in `problem`, change “9 coins” to “27 coins” and “Eight weigh the
same” to “26 weigh the same.” Keep one coin heavier. Rerun both calls. We already
know the minimum is **three weighings**: does the model give that answer, and does
its worked solution correctly narrow the candidates from **27 → 9 → 3 → 1**?

We asked for different kinds of output, so a longer second answer is expected.
What matters is whether its steps help you check the solution. Did they reveal
a mistake or cover a case you had overlooked? We'd need to repeat the comparison
on more problems before making a claim about improved accuracy.

### CoT prompting and reasoning effort are different controls

| Control | Where we change it | What we're asking for |
| --- | --- | --- |
| CoT prompt | The message text | A solution expressed through intermediate steps |
| Reasoning effort | A supported model's API setting | A change in the effort it allocates to internal reasoning |
| Reasoning summary | An optional supported response feature | A summary, rather than raw internal reasoning |

The distinction matters when reading the output. A short answer may follow
substantial internal reasoning, while a long worked solution may still contain
errors. Use the answer and reported usage to evaluate the result rather than
treating its length as a measure of reasoning quality.

OpenAI's [reasoning-model guidance](https://developers.openai.com/api/docs/guides/reasoning-best-practices)
recommends clear, direct prompts for reasoning models; adding “think step by step”
is generally unnecessary for them. We'll keep the next experiment's prompt direct
and vary the effort setting instead.

### Pause and reflect

Which response was easier to evaluate, and why? Did you find a missing case or
incorrect step? What would you need to test before concluding that this prompting
technique improves accuracy?

**Write your response here:**

Both calls agreed on the minimum which is 3 weighings for 27 coins. The worked solution's steps correctly reduced the candidate set 27->9->3->1 (weight 9v9, then 3v3 within the suspect group, then 1v1), matching the intended narrowing and correctly handling the "balanced" branch at each stage. It explicitly noted that a balanced result places the heavier coin in the untested group. The step-by-step answer was easier to evaluate specifically becase I could check its logic at each tage against my own reasoning, rather than just trusting a bare number. The direct answer (16 output tokens) gave no way to catch a mistake if the model had guessed wrong, while the worked version (250 output tokens) gave much more to verify, at a real token-cost tradeoff.

### G2. Optional: Compare Reasoning Effort with the Responses API

The last experiment changed the prompt. Some models also expose a
**reasoning-effort setting**, which gives us another way to influence how they
approach a task. Let's try that through the Responses API.

We'll use `gpt-5.6-luna`, which was tested through Cornell's gateway with both
`low` and `medium` reasoning effort.

A reasoning model is trained to work through problems using internal reasoning
before producing its visible answer. An effort setting influences how much
reasoning it uses; it doesn't promise a fixed duration or a correct solution.
That's why we compare both answer quality and resource use.

The API and the model each matter here. Responses defines the shape of our request
and the object we get back. The selected model determines whether a reasoning-effort
setting is supported. Choosing Responses alone doesn't add that capability.

### Read a Responses call

Open the [Responses API Reference](https://developers.openai.com/api/reference/python/resources/responses/methods/create)
and compare it with the calls we've made so far:

| Chat Completions example | Responses example |
| --- | --- |
| `messages` contains the conversation | `input` supplies text or supported input items; here, just our problem text |
| `completion.choices[0].message.content` reads the answer | `response.output_text` collects the returned text for display |

We'll compare `low` and `medium` effort on one model. The [reasoning guide](https://developers.openai.com/api/docs/guides/reasoning)
explains the setting; check the selected model for the values it supports.

### Choose the model and check the problem

The cell below sets `reasoning_model` to `"gpt-5.6-luna"`. Both calls will use
this model so that we can compare the effort settings.

We'll reuse `problem` from the CoT comparison. If you changed it to 27 coins,
that's what these calls will receive too. Read the printed problem before
continuing. This time, we ask for an answer and a brief strategy, without adding
the step-by-step instruction.

Each effort setting gets the same problem independently. We don't send the
low-effort answer into the medium-effort call. That lets us compare the setting
without giving one call an earlier solution to work from.

Run the cell to set the model and display the problem.

In [81]:
from time import perf_counter

reasoning_model = "gpt-5.6-luna"  # Model identifier tested through Cornell's gateway.
reasoning_input = problem + " Give the answer and a brief strategy."
print(reasoning_input)

You have 27 coins. Twenty-six weigh the same, and one is heavier. Using a balance scale, what is the minimum number of weighings needed in the worst case to identify the heavier coin? Give the answer and a brief strategy.


### First, try low reasoning effort

The `reasoning` argument sets the effort to `low`. We also record the time before
the call and subtract it from the time when the response arrives. That gives us
the elapsed time we experienced, including network and service delays.

`store=False` asks the Responses API not to store the response for later retrieval
through that API. After the call, we'll display the answer, status, and elapsed time.

Run the cell and inspect the answer before trying medium effort.

In [82]:
if reasoning_model:
    start = perf_counter()
    low_response = client.responses.create(
        model=reasoning_model,
        input=reasoning_input,
        reasoning={"effort": "low"},
        store=False,
    )
    low_seconds = perf_counter() - start

    display(Markdown(low_response.output_text))
    print("Status:", low_response.status)
    print("Elapsed seconds:", round(low_seconds, 2))

**Minimum: 3 weighings.**

Strategy:

1. Divide the 27 coins into three groups of 9. Weigh 9 against 9.  
   - If they balance, the heavier coin is in the unweighed group.  
   - Otherwise, it is in the heavier-weighing group.

2. From the suspected group of 9, divide into three groups of 3 and weigh 3 against 3. This identifies a group of 3 containing the heavier coin.

3. Weigh 1 coin against 1 from those 3.  
   - If one is heavier, it is the answer.  
   - If they balance, the third coin is heavier.

Thus, **3 weighings**, which is optimal since each weighing has three possible outcomes and \(3^3=27\).

Status: completed
Elapsed seconds: 3.23


### Now try medium reasoning effort

Change the effort to `medium` while keeping the problem, model, and other settings
the same. Before running the cell, predict what might change: the answer, the
explanation, the time it takes, or the token usage.

Both settings may solve the problem correctly. If they do, compare how useful
the answers are and what resources each call used.

In [83]:
if reasoning_model:
    start = perf_counter()
    medium_response = client.responses.create(
        model=reasoning_model,
        input=reasoning_input,
        reasoning={"effort": "medium"},
        store=False,
    )
    medium_seconds = perf_counter() - start

    display(Markdown(medium_response.output_text))
    print("Status:", medium_response.status)
    print("Elapsed seconds:", round(medium_seconds, 2))

**Minimum: 3 weighings.**

Strategy:

1. Divide the 27 coins into three groups of 9. Weigh 9 against 9. The heavier coin is in the heavier group, or in the third group if they balance.
2. Divide the suspect 9 into three groups of 3. Weigh 3 against 3 and select the heavier group, or the unweighed group if balanced.
3. Weigh 1 coin against 1 from the remaining 3. If they balance, the third is heavier; otherwise, the heavier side identifies it.

Three weighings are also necessary because each weighing has three possible outcomes, and \(3^2=9<27\), while \(3^3=27\).

Status: completed
Elapsed seconds: 3.62


### Compare the reported token usage

A reasoning model can generate reasoning tokens in addition to its final answer.
Those tokens contribute to reported output usage even though they aren't shown
as part of the answer text.

Run the next cell to compare output and reasoning-token counts. If your gateway
doesn't return these details, skip the cell and record that they were unavailable.

Check the response status too. If it is `incomplete`, inspect the reported reason
before treating the answer as a finished solution.

In [84]:
if reasoning_model:
    print("LOW EFFORT")
    print("Output tokens:", low_response.usage.output_tokens)
    print("Reasoning tokens:", low_response.usage.output_tokens_details.reasoning_tokens)

    print()
    print("MEDIUM EFFORT")
    print("Output tokens:", medium_response.usage.output_tokens)
    print("Reasoning tokens:", medium_response.usage.output_tokens_details.reasoning_tokens)

LOW EFFORT
Output tokens: 256
Reasoning tokens: 80

MEDIUM EFFORT
Output tokens: 283
Reasoning tokens: 123


### What should we compare?

Check each strategy against your own solution, then compare elapsed time and
usage. Did the higher effort setting help with this problem? If both answers
were useful, was one clearer or less costly in tokens?

Before looking at timing, check correctness. A fast wrong answer and a slower
correct answer are different outcomes; if both are correct, time and token usage
become useful ways to compare them. Our elapsed time includes network and service
delays, so a timing difference alone doesn't tell us how long the model reasoned.

Two calls give us an observation to discuss. We'd need repeated trials on more
problems before drawing a broader conclusion about the settings. The optional
reasoning summaries described earlier are a separate feature; this experiment
uses the final answers and reported usage.

### Pause and reflect

Did both responses solve the problem? Did the extra effort change the time or
token usage? If the answers were equally useful, which setting would you try first
for this task, and what further evidence would you want?

If you did not run the calls, write your prediction and label it as a prediction.

**Write your response here:**

Both effort settings solved the problem correctly (3 weightings) with essentially the same strategy. Medium effort used more of both output tokens (283 vs. 256) and reasoning tokens (123 vs. 80), and took longer (3.62s vs. 3.23s), for a problem that low effort already solved correctly and clearly. As such, for this problem, I'd pick low effort first, since the extra reasoning budget didn't produe a better or clearer answer, just a marginally longer one, so medium effort's cost wasn't justified here. I'd want to test both settings on a harder problem, like more coins or a trickier structure, before concluding low effort is always sufficient.

## H. Open the Standalone Streamlit Activity

We've been writing user messages in Python. Let's put a chat box around the same
pattern so someone can type a question and continue the conversation in a browser.

Open the [Streamlit guide](STREAMLIT.md), which explains the app line by line.
The app creates its own client and conversation history, so you can run it
independently of this notebook.

From the repository root, run:

```bash
streamlit run assignments/01-model-calls-and-chatbots/app.py
```

### From notebook cells to a running chat app

In the notebook, we choose which cell to run next. Streamlit runs `app.py` from
top to bottom, then runs it again when someone submits a message or clicks a
button. The app needs to keep the conversation available across those runs.
That's the job of `st.session_state.messages`.

Follow a submitted question through these steps in [app.py](app.py):

| Step | Where to look | What happens |
| --- | --- | --- |
| Read the input | `st.chat_input(...)` | Returns the text the user submitted. |
| Build the request | `request_messages = ...` | Combines saved history with the new user message. |
| Ask the model | `client.chat.completions.create(...)` | Sends that list and waits for the result. |
| Extract the answer | `response.choices[0].message.content` | Gets the text from the returned assistant message. |
| Display it | `st.markdown(answer)` | Shows the answer inside an assistant chat container. |
| Keep the exchange | `st.session_state.messages = ...` | Saves the request messages and assistant reply for the next turn. |

The app calls its returned object `response`; the notebook uses `completion`.
Both are variable names we chose, and both examples call Chat Completions.

Displaying an answer and saving it are separate steps. The display makes it
visible now; the saved history lets the app redraw it and include it in a later
request. This session state belongs to the running browser session, so it isn't
a permanent record of the conversation.

**Try a prediction:** If you kept the answer display but removed the final history
assignment, what would happen after the next message? Trace both the page display
and the next request before testing your prediction.

### Does this app remember, and does it stream?

It remembers the conversation **within the current Streamlit session** by saving
the message list and sending it with each new question. Clicking New conversation
clears that history. A browser reload or server restart also loses this temporary
state; the app doesn't write conversations to a file or database.

The provided app waits for the complete answer and then displays it. It **doesn't
stream**. The guide's streaming exercise changes the API request to return pieces
as they become available and displays those pieces as they arrive. The completed
answer still needs to be saved for the next turn.

The code has short comments to help you follow its main steps. Use the guide for
the detailed walkthrough and experiments with instructions, history, and streaming.

## Final Reflection

Choose one experiment from today and explain it using the request or code that
produced the result. You can use these questions to guide your response:

- Where does your chatbot's conversation history live?
- What do the system, user, and assistant roles contribute to a request?
- What changed when you changed a system prompt?
- How does supplied context differ from training knowledge?
- What did the CoT prompt make easier or harder to evaluate?
- What did the tokenizer show that a word count would not?
- What did you observe when comparing reasoning effort, if you ran that experiment?
- Why was inspecting the outgoing request useful?

**Write your response here:**

Without a source, `without_source` correctly declined to answer "What exact time of day was the 2024 Nobel Prize in Physics annouced?" The model said it didn't have the information, rather than guessing. This matcheed my expectation, since a specfic announcement time isn't the kind of fact a general knowledge cutoff would reliably encode even for events it does know about. After I supplied `source_text`, a paraphrase with only date, no time, the model still correctly refused to answer, saying the exact time "is not provided in the summary." This shows supplied context is different from training knowledge in an important way, particularly in giving the model a source doesn't make it invest facts to fill gaps, rather the source_instruction is what kept both responses honest, whether the missing information came from an untrained event or an incomplete document. Inspecting the actual request that was sent was useful earlier in the notebook for a different response, as it showed me that the model isn't reading my Python variables or notebook state, only the literal message list I contruct and pass to `messages=`.

## Save Your Work

Save the notebook with your observations and outputs. If you completed the separate
Streamlit activity, save your app changes as well. Follow the course README for
committing and pushing, and Canvas for submission requirements.

To check that the notebook runs from the beginning, restart the kernel and use
**Run All**. This sends real API requests, including the reasoning comparison.
To skip that comparison, set `reasoning_model = ""` before running; the `if`
statements in that section will skip its API calls.

A notebook's kernel holds its Python variables. Editing an earlier cell doesn't
update a value until you execute that cell, and old outputs can remain visible
after the code changes. Running from the beginning checks that the saved sequence
of cells produces the state your later examples need.

## Where to Learn More

- [Chat Completions API Reference](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create)
- [Responses API Reference](https://developers.openai.com/api/reference/python/resources/responses/methods/create)
- [OpenAI Model Spec (August 18, 2026 edition)](https://model-spec.openai.com/2026-08-18.html)
- [Conversation state and context](https://developers.openai.com/api/docs/guides/conversation-state)
- [Reasoning-model prompting guidance](https://developers.openai.com/api/docs/guides/reasoning-best-practices)
- [Reasoning models](https://developers.openai.com/api/docs/guides/reasoning)
- [Interactive tokenizer](https://platform.openai.com/tokenizer)